### Data Preparation (WS2 + WS4)

Extracts frames from two sets of videos:

- `videos/` → `data/<video_name>/`, one folder per video, ~1.5 s between frames, max 80 per video.
- `videos_1/` → `data/frames/`, all videos in one flat folder, ~3 s between frames, max 70 per video.

Blurry frames are skipped. A summary CSV lists how many frames were kept per video.

Outputs:
- `data/<video_name>/*.jpg`
- `data/frames/*.jpg`
- `data/frames_extraction_summary.csv`

#### videos_1

- filling_plant_bolivia
- Bolivia： FLEXSPEED for Side-Entry Valves (2014) (https://www.youtube.com/watch?v=PtKnRpGvPM4)
- Portugal： Petrogal LPG Cylinder Filling Plant (2013) (https://www.youtube.com/watch?v=UH-u0g3B8Pw)
- Automatic LPG Cylinder filling plant (https://www.youtube.com/shorts/sdo9Pg50GNY)
- LPG CYLINDER FILLING CAROUSEL (ELECTRONIC 36 sec.) (https://www.youtube.com/shorts/dvBFjJub6rE)
- LPG CYLINDER FILLING CAROUSEL (https://www.youtube.com/shorts/dvBFjJub6rE)
- LPG Cylinder Robotic Loading System (https://www.youtube.com/watch?v=TIVIFfRqbgU)

In [3]:
from pathlib import Path
import cv2
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

VIDEOS_DIR = ROOT / "videos"
VIDEOS_1_DIR = ROOT / "videos_1"
DATA_DIR = ROOT / "data"
FRAMES_DIR = DATA_DIR / "frames"
FRAMES_1_DIR = DATA_DIR / "frames_1"

VIDEO_EXT = {".mp4", ".mov", ".avi", ".mkv"}

DATA_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_DIR.mkdir(parents=True, exist_ok=True)
FRAMES_1_DIR.mkdir(parents=True, exist_ok=True)

print(f"ROOT:       {ROOT}")
print(f"videos/:    {VIDEOS_DIR}  exists={VIDEOS_DIR.exists()}")
print(f"videos_1/:  {VIDEOS_1_DIR}  exists={VIDEOS_1_DIR.exists()}")
print(f"data/:      {DATA_DIR}")

ROOT:       d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo
videos/:    d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\videos  exists=True
videos_1/:  d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\videos_1  exists=True
data/:      d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\data


### Frame extractor

One function handles both sets. Parameters:

- `out_dir` — where the JPGs go
- `seconds_between` — sample interval
- `max_frames` — hard cap per video
- `blur_threshold` — Laplacian variance below this is skipped
- `offsets` — optional list of fractions in [0, 1] to sample from multiple parts of the video

If `offsets` is empty, the extractor samples evenly across the whole video. If `offsets` is provided, it samples around each offset so the frames come from different parts of the clip rather than clustering at the start.

In [10]:
def extract_frames(
    video_path,
    out_dir,
    seconds_between=1.5,
    max_frames=80,
    blur_threshold=50,
    offsets=None,
    prefix=None,
    start_seconds=0,
):
    out_dir.mkdir(parents=True, exist_ok=True)

    cap = cv2.VideoCapture(str(video_path))
    if not cap.isOpened():
        print(f"[skip] {video_path.name}: cannot open")
        return 0, 0

    fps = cap.get(cv2.CAP_PROP_FPS) or 30.0
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    duration = total / fps if fps > 0 else 0
    step = max(1, int(round(fps * seconds_between)))
    
    # Calculate starting frame based on seconds
    start_frame = int(start_seconds * fps)

    # Build the list of frame indices to consider
    if offsets:
        candidate_indices = []
        per_offset = max(1, max_frames // len(offsets))
        window = int(seconds_between * fps) * per_offset
        for frac in offsets:
            center = int(frac * total)
            start = max(0, center - window // 2)
            end = min(total, center + window // 2)
            candidate_indices.extend(range(start, end, step))
        candidate_indices = sorted(set(candidate_indices))
    else:
        # Start from the specific frame, not 0
        candidate_indices = list(range(start_frame, total, step))

    name_prefix = prefix or video_path.stem
    saved, skipped_blur = 0, 0

    for frame_idx in candidate_indices:
        if saved >= max_frames:
            break

        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_idx)
        ok, frame = cap.read()
        if not ok:
            continue

        gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
        score = cv2.Laplacian(gray, cv2.CV_64F).var()

        if score < blur_threshold:
            skipped_blur += 1
            continue

        out_path = out_dir / f"{name_prefix}_{saved:04d}.jpg"
        cv2.imwrite(str(out_path), frame)
        saved += 1

    cap.release()
    return saved, skipped_blur

### Set 1 — `videos/` into `data/<video_name>/`

One folder per video. Sampling every ~1.5 s, max 80 frames per video.

To spread frames across different parts of the video instead of the first N seconds, set `USE_OFFSETS = True`. The `OFFSETS` list marks the fractions of the clip to sample around. For example `[0.1, 0.35, 0.6, 0.85]` samples four different regions.

If `USE_OFFSETS = False`, the extractor samples evenly from start to end.

In [12]:
SECONDS_BETWEEN = 1.5
MAX_FRAMES = 80
BLUR_THRESHOLD = 50
USE_OFFSETS = False
OFFSETS = [0.1, 0.35, 0.6, 0.85]
START_SECONDS = 300  # 5 minutes

video_files = sorted(
    p for p in VIDEOS_DIR.glob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXT
)

print(f"Found {len(video_files)} videos in {VIDEOS_DIR}")
print("=" * 60)

summary = []

for video in video_files:
    out_dir = DATA_DIR / video.stem
    saved, skipped = extract_frames(
        video,
        out_dir,
        seconds_between=SECONDS_BETWEEN,
        max_frames=MAX_FRAMES,
        blur_threshold=BLUR_THRESHOLD,
        offsets=OFFSETS if USE_OFFSETS else None,
        prefix=video.stem,
        start_seconds=START_SECONDS,  # Added here to skip the first 5 minutes
    )
    summary.append({
        "set": "videos",
        "video": video.name,
        "folder": str(out_dir.relative_to(ROOT)),
        "saved": saved,
        "skipped_blur": skipped,
    })
    print(f"{video.name:40s} saved={saved:3d}  skipped_blur={skipped:3d}")

print("=" * 60)

Found 6 videos in d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\videos
14_43_back_left_cropped.mp4              saved= 80  skipped_blur=  8
14_43_back_right_cropped.mp4             saved= 80  skipped_blur=  0
14_43_front_cropped.mp4                  saved= 80  skipped_blur=  0
14_55_back_left_cropped.mp4              saved= 80  skipped_blur=  1
14_55_back_right_cropped.mp4             saved= 80  skipped_blur=  3
14_55_front_cropped.mp4                  saved= 80  skipped_blur=  0


### Set 1B — `videos/` into `data/frames/`

All frames go into a single flat folder `data/frames/`.

Sampling every ~3 s, max 30 frames per video.

In [11]:
SECONDS_BETWEEN_1 = 1.5
MAX_FRAMES_1 = 50
BLUR_THRESHOLD_1 = 50
START_SECONDS_1 = 180 # 3 minutes

video_files = sorted(
    p for p in VIDEOS_DIR.glob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXT
)

print(f"Found {len(video_files)} videos in {VIDEOS_DIR}")
print("=" * 60)

for video in video_files:
    saved, skipped = extract_frames(
        video,
        FRAMES_DIR,
        seconds_between=SECONDS_BETWEEN_1,
        max_frames=MAX_FRAMES_1,
        blur_threshold=BLUR_THRESHOLD_1,
        prefix=video.stem,
        start_seconds=START_SECONDS_1, # Added here
    )
    summary.append({
        "set": "videos",
        "video": video.name,
        "folder": str(FRAMES_DIR.relative_to(ROOT)),
        "saved": saved,
        "skipped_blur": skipped,
    })
    print(f"{video.name:40s} saved={saved:3d}  skipped_blur={skipped:3d}")

print("=" * 60)

Found 6 videos in d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\videos
14_43_back_left_cropped.mp4              saved= 50  skipped_blur= 12
14_43_back_right_cropped.mp4             saved= 50  skipped_blur=  2
14_43_front_cropped.mp4                  saved= 50  skipped_blur=  1
14_55_back_left_cropped.mp4              saved= 50  skipped_blur=  1
14_55_back_right_cropped.mp4             saved= 50  skipped_blur= 10
14_55_front_cropped.mp4                  saved= 50  skipped_blur=  0


### Set 2 — `videos_1/` into `data/frames_1/`

All frames go into a single flat folder `data/frames_1/`.

Sampling every ~3 s, max 70 frames per video. Shorter videos will produce fewer frames automatically.

In [6]:
SECONDS_BETWEEN_2 = 3
MAX_FRAMES_2 = 30
BLUR_THRESHOLD_2 = 50

video_1_files = sorted(
    p for p in VIDEOS_1_DIR.glob("*")
    if p.is_file() and p.suffix.lower() in VIDEO_EXT
) if VIDEOS_1_DIR.exists() else []

print(f"Found {len(video_1_files)} videos in {VIDEOS_1_DIR}")
print("=" * 60)

for video in video_1_files:
    saved, skipped = extract_frames(
        video,
        FRAMES_1_DIR,
        seconds_between=SECONDS_BETWEEN_2,
        max_frames=MAX_FRAMES_2,
        blur_threshold=BLUR_THRESHOLD_2,
        prefix=video.stem,
    )
    summary.append({
        "set": "videos_1",
        "video": video.name,
        "folder": str(FRAMES_1_DIR.relative_to(ROOT)),
        "saved": saved,
        "skipped_blur": skipped,
    })
    print(f"{video.name:40s} saved={saved:3d}  skipped_blur={skipped:3d}")

print("=" * 60)

Found 7 videos in d:\BCS Files\Y3S1\AI Applications MBI38j\Labs\primagaz-yolo\videos_1
Automatic LPG Cylinder filling plant. #Shorts.mp4 saved=  6  skipped_blur=  0
Bolivia： FLEXSPEED for Side-Entry Valves (2014).mp4 saved= 53  skipped_blur=  2
filling_plant_bolivia.mp4                saved= 15  skipped_blur=  0
LPG CYLINDER FILLING CAROUSEL (ELECTRONIC 36 sec.).mp4 saved= 23  skipped_blur= 51
LPG CYLINDER FILLING CAROUSEL.mp4        saved= 17  skipped_blur=  1
LPG Cylinder Robotic Loading System.mp4  saved= 27  skipped_blur=  1
Portugal： Petrogal LPG Cylinder Filling Plant (2013).mp4 saved= 70  skipped_blur=  3


### Notes

- All JPGs are written at the original video resolution. If you want them smaller for CVAT, add a resize step in `extract_frames` before writing, e.g. `frame = cv2.resize(frame, (1280, 720))`.
- Blur threshold of 50 is a reasonable default. Lower it to 30 if too many frames are rejected, or raise to 80 to be stricter.
- For `videos/`, folders are `data/<video_name>/` so you can open one folder per video in CVAT.
- For `videos_1/`, everything is in `data/frames/`, prefixed with the source video name. Upload the whole folder to one CVAT task if you want all of them together.
- To sample from different parts of a long video, set `USE_OFFSETS = True` in the Set 1 cell. Adjust `OFFSETS` for how many regions you want.

### Opening CVAT offline

1. Open a terminal and Start CVAT

```bash
cd "path\to\cvat"
docker compose up -d
```
2. Open the web interface

http://localhost:8080

3. Stop CVAT when done
```bash
docker compose down
```

4. Exporting a dataset from CVAT

- Go to **Tasks** → click your task → **Menu → Export task dataset**.
- Choose format: **Ultralytics YOLO Detection 1.0**.
- Tick **Save images**.
- If you want a split, set **Train/Test split** to e.g. `80/20`. Otherwise leave it and everything goes to `train/`.
- Click **OK**.
- Go to **Requests** (top-right in the task page). Wait for the export to finish.
- Click the download icon and save the ZIP.
- If you are on a **Project** with multiple tasks, use **Menu → Export project dataset** instead. Same options.